In [35]:
import os
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump"
)

RAW_SOIL_DIR = PROJECT_ROOT / "data" / "raw" / "soil"
BOUNDARY_DIR = RAW_SOIL_DIR / "district_boundary"

PROCESSED_SOIL_DIR = (
    PROJECT_ROOT / "data" / "processed" / "soil"
)

UNIFIED_DIR = (
    PROJECT_ROOT / "data" / "processed" / "unified"
)

PROCESSED_SOIL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Raw soil:", RAW_SOIL_DIR)
print("Boundary:", BOUNDARY_DIR)
print("Processed soil:", PROCESSED_SOIL_DIR)

Project root: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump
Raw soil: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\raw\soil
Boundary: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\raw\soil\district_boundary
Processed soil: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\soil


In [36]:
#Verifying required files
SOIL_ZIPS = {
    "clayey": RAW_SOIL_DIR / "nices_isd_texture_clayey.zip",
    "clayey_skeletal": RAW_SOIL_DIR / "nices_isd_texture_clayskeletal.zip",
    "loamy": RAW_SOIL_DIR / "nices_isd_texture_loamy.zip",
    "sandy": RAW_SOIL_DIR / "nices_isd_texture_sandy.zip",
}

BOUNDARY_FILE = (
    BOUNDARY_DIR / "LGD_Districts.geojson"
)

print("Checking soil ZIP files...\n")

missing = []

for name, path in SOIL_ZIPS.items():
    if path.exists():
        print(f"✓ {name:20} {path.name}")
    else:
        print(f"✗ {name:20} MISSING")
        missing.append(path)

print("\nChecking district boundary...")

if BOUNDARY_FILE.exists():
    print(f"✓ {BOUNDARY_FILE}")
else:
    print(f"✗ MISSING: {BOUNDARY_FILE}")
    missing.append(BOUNDARY_FILE)

if missing:
    raise FileNotFoundError(
        "\nMissing required input files. "
        "Place all four soil ZIPs and LGD_Districts.geojson "
        "in the specified directories."
    )

print("\n✓ All required inputs are available.")

Checking soil ZIP files...

✓ clayey               nices_isd_texture_clayey.zip
✓ clayey_skeletal      nices_isd_texture_clayskeletal.zip
✓ loamy                nices_isd_texture_loamy.zip
✓ sandy                nices_isd_texture_sandy.zip

Checking district boundary...
✓ C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\raw\soil\district_boundary\LGD_Districts.geojson

✓ All required inputs are available.


In [37]:
#Extracting four soil datasets
EXTRACT_DIR = (
    RAW_SOIL_DIR / "extracted_texture"
)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for soil_type, zip_path in SOIL_ZIPS.items():

    output_dir = EXTRACT_DIR / soil_type
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(output_dir)

    print(f" Extracted {soil_type}")

print("\nExtraction complete.")

 Extracted clayey
 Extracted clayey_skeletal
 Extracted loamy
 Extracted sandy

Extraction complete.


In [38]:
#Loacting and reading all 4 soil rasters
ASC_FILES = {}
SOIL_ARRAYS = {}
RASTER_META = {}

for soil_type in SOIL_ZIPS.keys():

    matches = list(
        (EXTRACT_DIR / soil_type).rglob("*.asc")
    )

    if not matches:
        raise FileNotFoundError(
            f"No ASC raster found for {soil_type}"
        )

    asc_file = matches[0]
    ASC_FILES[soil_type] = asc_file

    with rasterio.open(asc_file) as src:

        arr = src.read(1).astype("float64")

        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        SOIL_ARRAYS[soil_type] = arr

        RASTER_META[soil_type] = {
            "width": src.width,
            "height": src.height,
            "crs": src.crs,
            "transform": src.transform,
            "resolution": src.res,
            "nodata": src.nodata,
        }

    print(
        f"✓ {soil_type:20} "
        f"shape={arr.shape} "
        f"resolution={RASTER_META[soil_type]['resolution']}"
    )

✓ clayey               shape=(694, 625) resolution=(5000.0, 5000.0)
✓ clayey_skeletal      shape=(694, 625) resolution=(5000.0, 5000.0)
✓ loamy                shape=(694, 625) resolution=(5000.0, 5000.0)
✓ sandy                shape=(694, 625) resolution=(5000.0, 5000.0)


In [39]:
#Validating raster grids + convert fractions
reference = RASTER_META["clayey"]

for soil_type, meta in RASTER_META.items():

    assert meta["width"] == reference["width"], (
        f"Width mismatch: {soil_type}"
    )

    assert meta["height"] == reference["height"], (
        f"Height mismatch: {soil_type}"
    )

    assert meta["crs"] == reference["crs"], (
        f"CRS mismatch: {soil_type}"
    )

    assert meta["transform"] == reference["transform"], (
        f"Transform mismatch: {soil_type}"
    )

print("✓ All four rasters use the same grid.")

# converting integer scaled values to fractions
SOIL_FRACTIONS = {}

for soil_type, arr in SOIL_ARRAYS.items():

    # Official NRSC/Bhuvan texture rasters use integer-scaled fractional values.
    fraction = arr / 10000.0

    SOIL_FRACTIONS[soil_type] = fraction

print("\n✓ Soil raster values converted to fractions.")

✓ All four rasters use the same grid.

✓ Soil raster values converted to fractions.


In [40]:
#Soil raster quality check
for soil_type, arr in SOIL_FRACTIONS.items():

    valid = arr[np.isfinite(arr)]

    print(
        f"{soil_type:20} "
        f"min={valid.min():.4f} | "
        f"max={valid.max():.4f} | "
        f"mean={valid.mean():.4f}"
    )

    if valid.min() < 0 or valid.max() > 1:
        raise ValueError(
            f"Invalid fraction range in {soil_type}"
        )

# Check combined texture fraction
stack = np.stack(
    [
        SOIL_FRACTIONS["clayey"],
        SOIL_FRACTIONS["clayey_skeletal"],
        SOIL_FRACTIONS["loamy"],
        SOIL_FRACTIONS["sandy"],
    ]
)

valid_mask = np.all(
    np.isfinite(stack),
    axis=0
)

texture_sum = stack[:, valid_mask].sum(axis=0)

deviation = np.abs(texture_sum - 1)

print("\nTexture fraction sum:")
print("Mean deviation   :", deviation.mean())
print("Median deviation :", np.median(deviation))
print("Maximum deviation:", deviation.max())

print(
    "\nCells within ±0.01:",
    round(np.mean(deviation <= 0.01) * 100, 2),
    "%"
)

print("✓ Raster quality check complete.")

clayey               min=0.0000 | max=1.0000 | mean=0.0925
clayey_skeletal      min=0.0000 | max=1.0000 | mean=0.0136
loamy                min=0.0000 | max=1.0000 | mean=0.1129
sandy                min=0.0000 | max=1.0000 | mean=0.0181

Texture fraction sum:
Mean deviation   : 0.7636242778097982
Median deviation : 1.0
Maximum deviation: 1.0

Cells within ±0.01: 5.19 %
✓ Raster quality check complete.


In [43]:
# Cell 8 — Load district boundaries

import geopandas as gpd
from pathlib import Path

# Path to district boundary file
district_path = Path(
    r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump"
    r"\data\raw\soil\district_boundary\IND_ADM2.geojson"
)

# Check file
print("File exists:", district_path.exists())
print("File size (MB):", round(district_path.stat().st_size / (1024**2), 2))

# Load GeoJSON
districts = gpd.read_file(district_path)

print("\nDistrict boundary dataset loaded successfully!")
print("Shape:", districts.shape)
print("CRS:", districts.crs)

print("\nColumns:")
print(districts.columns.tolist())

print("\nFirst 3 records:")
display(districts.head(3))

File exists: True
File size (MB): 46.08

District boundary dataset loaded successfully!
Shape: (735, 6)
CRS: EPSG:4326

Columns:
['shapeName', 'shapeISO', 'shapeID', 'shapeGroup', 'shapeType', 'geometry']

First 3 records:


,shapeName,shapeISO,shapeID,shapeGroup,shapeType,geometry
0,Ashoknagar,,76128533B75548370501185,IND,ADM2,"POLYGON ((78.17491 24.84254, 78.17466 24.84194..."
1,Raisen,,76128533B57893545331548,IND,ADM2,"POLYGON ((77.38167 23.07004, 77.38094 23.0698,..."
2,Chhindwara,,76128533B70646408240587,IND,ADM2,"POLYGON ((79.23988 22.79135, 79.24 22.79172, 7..."


In [44]:
# Inspecting district boundary fields

print("Columns:")
print(districts.columns.tolist())

print("\nNumber of districts:", len(districts))

print("\nSample districts:")
display(districts[["shapeName", "shapeISO", "shapeID", "geometry"]].head())

# The geojason file provides district names through shapeName.
DISTRICT_FIELD = "shapeName"

# No state field is present in this boundary dataset.
STATE_FIELD = None

print("\nDistrict field:", DISTRICT_FIELD)
print("State field   :", STATE_FIELD)

Columns:
['shapeName', 'shapeISO', 'shapeID', 'shapeGroup', 'shapeType', 'geometry']

Number of districts: 735

Sample districts:


,shapeName,shapeISO,shapeID,geometry
0,Ashoknagar,,76128533B75548370501185,"POLYGON ((78.17491 24.84254, 78.17466 24.84194..."
1,Raisen,,76128533B57893545331548,"POLYGON ((77.38167 23.07004, 77.38094 23.0698,..."
2,Chhindwara,,76128533B70646408240587,"POLYGON ((79.23988 22.79135, 79.24 22.79172, 7..."
3,Betul,,76128533B82559220423608,"POLYGON ((78.27229 22.39973, 78.27169 22.39897..."
4,Hoshangabad,,76128533B45314020251888,"POLYGON ((78.03027 22.79953, 78.02986 22.79969..."



District field: shapeName
State field   : None


In [ ]:
#Loading soil raster CRS and reprojecting district boundaries

from pathlib import Path
import rasterio

SOIL_RAW = Path(
    r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump"
    r"\data\raw\soil"
)

# Locating the Clayey raster
clayey_rasters = list(SOIL_RAW.rglob("fclayey.asc"))

if not clayey_rasters:
    raise FileNotFoundError(
        "fclayey.asc not found inside the soil raw-data folder."
    )

clayey_path = clayey_rasters[0]

# Reading CRS directly from the raster
with rasterio.open(clayey_path) as src:
    soil_crs = src.crs

print("Clayey raster:", clayey_path)
print("Soil raster CRS:", soil_crs)

print("Original district CRS:", districts.crs)

# Reprojecting district boundaries
districts = districts.to_crs(soil_crs)

print("Reprojected district CRS:", districts.crs)

print("\nReprojection completed successfully.")

Clayey raster: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\raw\soil\texture\fclayey.asc
Soil raster CRS: PROJCS["unnamed",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",20],PARAMETER["longitude_of_center",78],PARAMETER["standard_parallel_1",28],PARAMETER["standard_parallel_2",12],PARAMETER["false_easting",2000000],PARAMETER["false_northing",2000000],UNIT["METERS",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Original district CRS: EPSG:4326
Reprojected district CRS: PROJCS["unnamed",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.01745329251

In [47]:
# Validating district geometries

print("Total districts:", len(districts))

print("Empty geometries:",
      districts.geometry.is_empty.sum())

print("Missing geometries:",
      districts.geometry.isna().sum())

print("Invalid geometries:",
      (~districts.geometry.is_valid).sum())

print("Duplicate district names:",
      districts["shapeName"].duplicated().sum())

# Fixing invalid geometries if any
if (~districts.geometry.is_valid).any():
    districts["geometry"] = districts.geometry.make_valid()

print("\nGeometry validation completed.")

Total districts: 735
Empty geometries: 0
Missing geometries: 0
Invalid geometries: 1
Duplicate district names: 7

Geometry validation completed.


In [48]:
# Loading soil texture rasters

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask

SOIL_RAW = Path(
    r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump"
    r"\data\raw\soil"
)

asc_files = {
    "clayey": next(SOIL_RAW.rglob("fclayey.asc")),
    "clayey_skeletal": next(SOIL_RAW.rglob("fclayskeletal.asc")),
    "loamy": next(SOIL_RAW.rglob("floamy.asc")),
    "sandy": next(SOIL_RAW.rglob("fsandy.asc")),
}

soil_arrays = {}
soil_transform = None
soil_crs = None
soil_height = None
soil_width = None

for texture, path in asc_files.items():

    with rasterio.open(path) as src:

        arr = src.read(1).astype(float)

        # Convert NoData to NaN
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        # NRSC stores fractional area as integer × 10,000
        arr = arr / 10000.0

        soil_arrays[texture] = arr

        if soil_transform is None:
            soil_transform = src.transform
            soil_crs = src.crs
            soil_height = src.height
            soil_width = src.width

        # Check that all rasters use the same grid
        assert src.width == soil_width
        assert src.height == soil_height
        assert src.transform == soil_transform
        assert src.crs == soil_crs

print("Soil rasters loaded successfully.")
print("Grid size:", soil_width, "×", soil_height)
print("CRS:", soil_crs)
print("Cell size:", soil_transform.a, "m")
print("\nRaster files:")

for name, path in asc_files.items():
    print(f"{name:20s} -> {path}")

Soil rasters loaded successfully.
Grid size: 625 × 694
CRS: PROJCS["unnamed",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",20],PARAMETER["longitude_of_center",78],PARAMETER["standard_parallel_1",28],PARAMETER["standard_parallel_2",12],PARAMETER["false_easting",2000000],PARAMETER["false_northing",2000000],UNIT["METERS",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Cell size: 5000.0 m

Raster files:
clayey               -> C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\raw\soil\texture\fclayey.asc
clayey_skeletal      -> C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\raw\soil\texture\fclayskeletal.asc
loamy                -> C:\Users\omrka\Documents\USB\

In [49]:
# Calculating district-level soil texture fractions

records = []

texture_names = [
    "clayey",
    "clayey_skeletal",
    "loamy",
    "sandy"
]

display_names = {
    "clayey": "Clayey",
    "clayey_skeletal": "Clayey skeletal",
    "loamy": "Loamy",
    "sandy": "Sandy"
}

for idx, row in districts.iterrows():

    geom = row.geometry

    if geom is None or geom.is_empty:
        continue

    # Identifying 5-km raster cells whose centers fall inside the district
    mask = geometry_mask(
        [geom],
        transform=soil_transform,
        invert=True,
        out_shape=(soil_height, soil_width),
        all_touched=False
    )

    values = {}

    for texture in texture_names:

        arr = soil_arrays[texture]

        selected = arr[mask]
        selected = selected[np.isfinite(selected)]

        if len(selected) > 0:
            values[texture] = float(np.mean(selected))
        else:
            values[texture] = np.nan

    fractions = [
        values["clayey"],
        values["clayey_skeletal"],
        values["loamy"],
        values["sandy"]
    ]

    # Determine dominant soil texture
    valid = [
        (texture, value)
        for texture, value in zip(texture_names, fractions)
        if np.isfinite(value)
    ]

    if valid:
        dominant_texture = max(valid, key=lambda x: x[1])[0]
        dominant_soil = display_names[dominant_texture]
    else:
        dominant_soil = np.nan

    records.append({
        "district": row["shapeName"],
        "clayey_fraction": values["clayey"],
        "clayey_skeletal_fraction": values["clayey_skeletal"],
        "loamy_fraction": values["loamy"],
        "sandy_fraction": values["sandy"],
        "soil_type": dominant_soil
    })

soil_district = pd.DataFrame(records)

print("District-level soil processing completed.")
print("Shape:", soil_district.shape)

display(soil_district.head(10))

District-level soil processing completed.
Shape: (735, 6)


,district,clayey_fraction,clayey_skeletal_fraction,loamy_fraction,sandy_fraction,soil_type
0,Ashoknagar,0.842491,0.000000,0.047166,0.000000,Clayey
1,Raisen,0.794527,0.014597,0.023989,0.000000,Clayey
2,Chhindwara,0.408950,0.000000,0.512497,0.000000,Loamy
3,Betul,0.394017,0.000000,0.557140,0.000000,Loamy
4,Hoshangabad,0.631622,0.000000,0.165245,0.000000,Clayey
5,Sehore,0.590613,0.178713,0.122048,0.010668,Clayey
6,Jabalpur,0.656665,0.000000,0.253126,0.000000,Clayey
7,Narsimhapur,0.741278,0.000000,0.185214,0.000000,Clayey
8,Panna,0.513915,0.000000,0.437548,0.018505,Clayey
9,Ujjain,0.907455,0.028734,0.012048,0.000000,Clayey


In [50]:
# Validating district soil dataset

fraction_cols = [
    "clayey_fraction",
    "clayey_skeletal_fraction",
    "loamy_fraction",
    "sandy_fraction"
]

print("SOIL DATA VALIDATION")

print("\nTotal district records:")
print(len(soil_district))

print("\nUnique district names:")
print(soil_district["district"].nunique())

print("\nDuplicate district names:")
print(soil_district["district"].duplicated().sum())

print("\nMissing values:")
print(soil_district[fraction_cols + ["soil_type"]].isna().sum())

# Calculate total texture fraction
soil_district["fraction_sum"] = soil_district[
    fraction_cols
].sum(axis=1, min_count=1)

print("\nFraction-sum statistics:")
display(soil_district["fraction_sum"].describe())

print("\nSoil type distribution:")
display(
    soil_district["soil_type"]
    .value_counts(dropna=False)
)

print("\nDistricts with no soil information:")
print(
    soil_district["fraction_sum"].isna().sum()
)

print("\nSample processed records:")
display(soil_district.head(15))

SOIL DATA VALIDATION

Total district records:
735

Unique district names:
728

Duplicate district names:
7

Missing values:
clayey_fraction             1
clayey_skeletal_fraction    1
loamy_fraction              1
sandy_fraction              1
soil_type                   1
dtype: int64

Fraction-sum statistics:


count    734.000000
mean       0.779602
std        0.192349
min        0.000000
25%        0.714262
50%        0.838369
75%        0.910139
max        0.994699
Name: fraction_sum, dtype: float64


Soil type distribution:


soil_type
Loamy              407
Clayey             284
Clayey skeletal     27
Sandy               16
NaN                  1
Name: count, dtype: int64


Districts with no soil information:
1

Sample processed records:


,district,clayey_fraction,clayey_skeletal_fraction,loamy_fraction,sandy_fraction,soil_type,fraction_sum
0,Ashoknagar,0.842491,0.000000,0.047166,0.000000,Clayey,0.889658
1,Raisen,0.794527,0.014597,0.023989,0.000000,Clayey,0.833113
2,Chhindwara,0.408950,0.000000,0.512497,0.000000,Loamy,0.921446
3,Betul,0.394017,0.000000,0.557140,0.000000,Loamy,0.951157
4,Hoshangabad,0.631622,0.000000,0.165245,0.000000,Clayey,0.796867
5,Sehore,0.590613,0.178713,0.122048,0.010668,Clayey,0.902042
6,Jabalpur,0.656665,0.000000,0.253126,0.000000,Clayey,0.909791
7,Narsimhapur,0.741278,0.000000,0.185214,0.000000,Clayey,0.926491
8,Panna,0.513915,0.000000,0.437548,0.018505,Clayey,0.969968
9,Ujjain,0.907455,0.028734,0.012048,0.000000,Clayey,0.948237


In [52]:
# Checking compatibility with unified crop-yield dataset

ROOT = Path(
    r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump"
)

crop_file = (
    ROOT
    / "data"
    / "processed"
    / "unified"
    / "unified_crop_yield_2013_2025.csv"
)

if not crop_file.exists():
    raise FileNotFoundError(
        f"Unified crop dataset not found:\n{crop_file}"
    )

crop_keys = pd.read_csv(
    crop_file,
    usecols=["state", "district"]
).drop_duplicates()

print("===== CROP ↔ SOIL DISTRICT CHECK =====")

print(
    "Unique crop state-district combinations:",
    len(crop_keys)
)

print(
    "Unique crop district names:",
    crop_keys["district"].nunique()
)

print(
    "Soil district records:",
    len(soil_district)
)

print(
    "Unique soil district names:",
    soil_district["district"].nunique()
)

# Find district names occurring in multiple states
district_state_counts = (
    crop_keys
    .groupby("district")["state"]
    .nunique()
    .reset_index(name="state_count")
)

ambiguous = district_state_counts[
    district_state_counts["state_count"] > 1
]

print(
    "\nDistrict names occurring in multiple states:",
    len(ambiguous)
)

if len(ambiguous) > 0:

    print("\nExamples of ambiguous district names:")

    display(
        crop_keys[
            crop_keys["district"].isin(
                ambiguous["district"]
            )
        ]
        .sort_values(["district", "state"])
        .head(30)
    )

# Exact district-name overlap
crop_districts = set(
    crop_keys["district"].dropna().astype(str).str.strip()
)

soil_districts = set(
    soil_district["district"].dropna().astype(str).str.strip()
)

matched = crop_districts & soil_districts

crop_without_soil = crop_districts - soil_districts
soil_without_crop = soil_districts - crop_districts

print("\nDistrict-name overlap:")
print("Matched district names:", len(matched))

print(
    "Crop district names without soil:",
    len(crop_without_soil)
)

print(
    "Soil district names without crop data:",
    len(soil_without_crop)
)

if crop_without_soil:
    print("\nSample crop districts without soil:")
    print(sorted(crop_without_soil)[:30])


===== CROP ↔ SOIL DISTRICT CHECK =====
Unique crop state-district combinations: 806
Unique crop district names: 803
Soil district records: 735
Unique soil district names: 728

District names occurring in multiple states: 3

Examples of ambiguous district names:


,state,district
742,Chhattisgarh,Bilaspur
1305,Himachal Pradesh,Bilaspur
1317,Himachal Pradesh,Hamirpur
4363,Uttar Pradesh,Hamirpur
3531,Rajasthan,Pratapgarh
4613,Uttar Pradesh,Pratapgarh



District-name overlap:
Matched district names: 617
Crop district names without soil: 186
Soil district names without crop data: 111

Sample crop districts without soil:
['Agar Malwa', 'Agar-Malwa', 'Ahilyanagar', 'Ahmedabad', 'Alluri Sitharama Raju', 'Amroha', 'Anakapalli', 'Ananthapuramu', 'Angul', 'Annamayya', 'Arvalli', 'Ayodhya', 'Bagalkote', 'Bajali', 'Balasore', 'Ballari', 'Balodabazar Bhatapara', 'Balodabazar-Bhatapara', 'Balotra', 'Balrampur Ramanujganj', 'Balrampur-Ramanujganj', 'Bandipora', 'Bapatla', 'Baramulla', 'Beawar', 'Beed', 'Belagavi', 'Bemetara', 'Bengaluru Rural', 'Bengaluru South']


In [53]:
# Saving final district-level soil dataset

PROCESSED_SOIL = (
    ROOT
    / "data"
    / "processed"
    / "soil"
)

PROCESSED_SOIL.mkdir(
    parents=True,
    exist_ok=True
)

output = (
    PROCESSED_SOIL
    / "district_soil_texture_nrsc_5km.csv"
)

# Remove validation-only column
final_soil = soil_district.drop(
    columns=["fraction_sum"],
    errors="ignore"
)

final_soil.to_csv(
    output,
    index=False
)

print("FINAL SOIL DATASET SAVED")

print("\nFile:")
print(output)

print("\nShape:")
print(final_soil.shape)

print("\nColumns:")
print(final_soil.columns.tolist())

print("\nMissing values:")
print(final_soil.isna().sum())

print("\nDuplicate district names:")
print(
    final_soil["district"].duplicated().sum()
)

print("\nSoil type distribution:")
display(
    final_soil["soil_type"]
    .value_counts(dropna=False)
)

print("\nPreview:")
display(final_soil.head(10))

FINAL SOIL DATASET SAVED

File:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\soil\district_soil_texture_nrsc_5km.csv

Shape:
(735, 6)

Columns:
['district', 'clayey_fraction', 'clayey_skeletal_fraction', 'loamy_fraction', 'sandy_fraction', 'soil_type']

Missing values:
district                    0
clayey_fraction             1
clayey_skeletal_fraction    1
loamy_fraction              1
sandy_fraction              1
soil_type                   1
dtype: int64

Duplicate district names:
7

Soil type distribution:


soil_type
Loamy              407
Clayey             284
Clayey skeletal     27
Sandy               16
NaN                  1
Name: count, dtype: int64


Preview:


,district,clayey_fraction,clayey_skeletal_fraction,loamy_fraction,sandy_fraction,soil_type
0,Ashoknagar,0.842491,0.000000,0.047166,0.000000,Clayey
1,Raisen,0.794527,0.014597,0.023989,0.000000,Clayey
2,Chhindwara,0.408950,0.000000,0.512497,0.000000,Loamy
3,Betul,0.394017,0.000000,0.557140,0.000000,Loamy
4,Hoshangabad,0.631622,0.000000,0.165245,0.000000,Clayey
5,Sehore,0.590613,0.178713,0.122048,0.010668,Clayey
6,Jabalpur,0.656665,0.000000,0.253126,0.000000,Clayey
7,Narsimhapur,0.741278,0.000000,0.185214,0.000000,Clayey
8,Panna,0.513915,0.000000,0.437548,0.018505,Clayey
9,Ujjain,0.907455,0.028734,0.012048,0.000000,Clayey
